# Concrete strength prediction

This notebook provides a reproducible regression workflow for the Yeh concrete-strength dataset. The dataset is not bundled with this repository. Run the notebook from the repository root, or set `CONCRETE_DATA_PATH` to a CSV file before loading it.


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from concrete_strength import FEATURES, RANDOM_STATE, TARGET, load_dataset

DEFAULT_DATA_PATH = Path("data") / "Concrete_Data_Yeh.csv"
DATA_PATH = Path(os.environ.get("CONCRETE_DATA_PATH", DEFAULT_DATA_PATH)).expanduser()


In [ ]:
df = load_dataset(DATA_PATH)
df.head()


In [ ]:
# Basic data checks are performed before any train/test split.
df.describe().T
print(f"Rows: {len(df):,}; features: {len(FEATURES)}; target: {TARGET}")


## Exploratory plots

These plots are descriptive only; no test-set information is used to fit a model.


In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=df, x=TARGET, bins=30, kde=True, ax=axes[0], color="firebrick")
axes[0].set_title("Distribution of concrete strength")
sns.scatterplot(data=df, x="age", y=TARGET, ax=axes[1], color="steelblue")
axes[1].set_title("Age versus concrete strength")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(df[FEATURES + [TARGET]].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Feature and target correlations")
plt.tight_layout()
plt.show()


## Reproducible model comparison

The split is made before fitting transformations. The linear model uses a `Pipeline`, so scaling parameters are learned from the training data only. The reported metrics are computed once on the held-out test set; they are not a substitute for cross-validation.


In [ ]:
X = df[FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

models = {
    "linear_regression": make_pipeline(StandardScaler(), LinearRegression()),
    "random_forest": RandomForestRegressor(
        n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
    ),
}

results = []
predictions = {}
for name, estimator in models.items():
    estimator.fit(X_train, y_train)
    prediction = estimator.predict(X_test)
    predictions[name] = prediction
    results.append(
        {
            "model": name,
            "r2": r2_score(y_test, prediction),
            "mae": mean_absolute_error(y_test, prediction),
            "rmse": np.sqrt(mean_squared_error(y_test, prediction)),
        }
    )

results_df = pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)
results_df


In [ ]:
best_model_name = results_df.loc[0, "model"]
comparison = pd.DataFrame(
    {"actual": y_test.to_numpy(), "predicted": predictions[best_model_name]}
)
comparison.head(10)
